In [ ]:
# Some plate for IMTM give repeated failures
# Analyze with correlations over plates instead of just the positive controls
# Also look deeper into available data

# Offending plates are: C1288 and C1287 with all 4 replicates affected:
## Batches:
### Plates

## 251113
### C1287_R3
### C1288_R3

## 251118
### C1283_R2
### C1284_R2
### C1285_R2
### C1287_R2
### C1288_R2

## 251210
### C1287_R4
### C1288_R4

## 251212
### C1287_R1
### C1288_R1

In [ ]:
import pandas as pd
import os
import glob
import CBE_utils as CBE
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import importlib
importlib.reload(CBE)

In [ ]:
input_path = "/media/schmied.christopher/T7 Shield/Datasets/ECBL/processed/"
output_path = input_path

results_path = "/home/schmied.christopher/FMP_Docs/Projects/eu_os_ecbl_qc/results/"

In [ ]:
site_specific_path = os.path.join(input_path, "IMTM")
     
# load raw data
pattern = "[A-Z]*_R[1-4].csv"
file_list = glob.glob(site_specific_path + os.sep + '*' + os.sep + pattern, recursive=True)

Data = []
    
for file in file_list:
    
    plate_map_name = os.path.splitext(os.path.basename(file))[0]
    
    try:
        
        Data_Temp = pd.read_csv(file)
        row_count = Data_Temp.shape[0]
        
        print(f"File: {plate_map_name} has {row_count} rows")
        
        Data.append(Data_Temp)
            
    except Exception as e:
        
        print(f"Error reading file {plate_map_name}: {e}")
        
        
### concat all files together
Data_aggregated = pd.concat(Data)
Data_aggregated = Data_aggregated.reset_index(drop = True)

print("Aggregated Data has shape ", Data_aggregated.shape)

In [ ]:
Data_aggregated.head()

In [ ]:
for cols in Data_aggregated.columns:
    print(cols)

In [ ]:
# Sanity check via plate map over cell count
# Lets have a look at the object counts...
# Could give an indication on if the right randomization scheme has been used. 
# It would be good to compare this with some other plates.
# Would be good to also look at the original images and the segmentations...

In [ ]:
cols_to_keep = [
    "Metadata_Batch",
    "Metadata_staining_date",
    "Metadata_Plate",
    "Metadata_Well_randomized",
    "Metadata_Object_Count",
    "Metadata_plate_name",
    "Metadata_replicate_number",
    "Metadata_plate_map_name",
    "Metadata_Well",
    "Metadata_RoughID"
]

Data_aggregated_reduced = Data_aggregated[cols_to_keep].copy()

In [ ]:
# get good min max for the same color scale across all tested plates
color_min = Data_aggregated_reduced["Metadata_Object_Count"].min()
color_max = Data_aggregated_reduced["Metadata_Object_Count"].max()

In [ ]:
# get row and column values 
Data_aggregated_reduced_copy = Data_aggregated_reduced.copy()

# Well like "A01" -> row="A", col=1
Data_aggregated_reduced_copy["row"] = Data_aggregated_reduced_copy["Metadata_Well"].str[0]                 # "A"
Data_aggregated_reduced_copy["col"] = Data_aggregated_reduced_copy["Metadata_Well"].str[1:].astype(int)    # "01" -> 1

In [ ]:
Data_aggregated_reduced_copy.head()

In [ ]:
def plot_plate_heatmaps(df, platename, vmin, vmax):

    df = df[df["Metadata_plate_name"] == platename]
    
    replicates = ["R1", "R2", "R3", "R4"]

    row_order = list("ABCDEFGHIJKLMNOP")          # A..P (16)
    col_order = list(range(1, 25))   

    fig, axes = plt.subplots(1, 4, figsize=(16, 3), constrained_layout=True)
    axes = axes.ravel()

    for ax, rep in zip(axes, replicates):

        drep = df[df["Metadata_replicate_number"] == rep]

        grid = (
            drep.pivot(index="row", columns="col", values="Metadata_Object_Count")
            ).reindex(index=row_order, columns=col_order)

        im = ax.imshow(grid.to_numpy(), vmin=vmin, vmax=vmax, aspect="auto", origin="upper")
        ax.set_title(f"Metadata_Object_Count — {rep}")

        ax.set_yticks(range(len(row_order)))
        ax.set_yticklabels(row_order)

        ax.set_xticks(range(len(col_order)))
        ax.set_xticklabels([f"{c:02d}" for c in col_order], rotation=90)

    cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.85)
    cbar.set_label("Metadata_Object_Count")
    plt.show()

In [ ]:
batches_to_keep = ["2025-10-22", "2025-11-13", "2025-11-18", "2025-12-10", "2025-12-12"]

Data_aggregated_reduced_copy_filtered = Data_aggregated_reduced_copy[Data_aggregated_reduced_copy["Metadata_staining_date"].isin(batches_to_keep)]

In [ ]:
plot_plate_heatmaps(Data_aggregated_reduced_copy_filtered, "C1287", color_min, color_max)

In [ ]:
plot_plate_heatmaps(Data_aggregated_reduced_copy_filtered, "C1288", color_min, color_max)

In [ ]:
plot_plate_heatmaps(Data_aggregated_reduced_copy_filtered, "C1283", color_min, color_max)

In [ ]:
plot_plate_heatmaps(Data_aggregated_reduced_copy_filtered, "C1284", color_min, color_max)

In [ ]:
# A naive guess could be that the positive controls have been incorrectely pipetted into the mother plate
# In C1287, C1288 they might be in A23 and A24 for nocodazole. 
# Or the nocodazole has not been properly dispensed to B24
# This could be this dreaded leakage problem also...

In [ ]:
# 251212
# C1287
# R1

filtered = Data_aggregated[(Data_aggregated["Metadata_staining_date"] == "2025-12-12") & (Data_aggregated["Metadata_plate_name"] == "C1287")  & (Data_aggregated["Metadata_replicate_number"] == "R1")]

In [ ]:
filtered2 = filtered [
    filtered ["Metadata_Well"].isin(["A23", "A24", "B23", "B24", "C23", "C24", "D23", "D24", "E23", "E24", "F23", "F24"]) 
]

In [ ]:
filtered2[["Metadata_Well", "Metadata_RoughID"]]

In [ ]:
metadata_cols = [c for c in filtered2.columns if c.startswith("Metadata_")]
feature_cols = [c for c in filtered2.columns if not c.startswith("Metadata_")]

well_pairs = [
    ("A23", "A24"),
    ("A24", "B24"),
]

In [ ]:
w1 = "A23"
w2 = "A24"

v1 = filtered2.loc[filtered2["Metadata_Well"] == w1, feature_cols].iloc[0]
v2 = filtered2.loc[filtered2["Metadata_Well"] == w2, feature_cols].iloc[0]

corr = v1.corr(v2)
print(corr)

In [ ]:
w1 = "A24"
w2 = "B24"

v1 = filtered2.loc[filtered2["Metadata_Well"] == w1, feature_cols].iloc[0]
v2 = filtered2.loc[filtered2["Metadata_Well"] == w2, feature_cols].iloc[0]

corr = v1.corr(v2)
print(corr)

In [ ]:
w1 = "F23"
w2 = "F24"

v1 = filtered2.loc[filtered2["Metadata_Well"] == w1, feature_cols].iloc[0]
v2 = filtered2.loc[filtered2["Metadata_Well"] == w2, feature_cols].iloc[0]

corr = v1.corr(v2)
print(corr)

In [ ]:
# The issue is now that I need to perform 
# Normalization using save DMSO wells
# Plus the feature reduction on these to get useful correlation values.
# Also it would be good to look into older plates
# To confirm that this is indeed the case with these patterns in persumably good plates

# Control wells

A23-P23	DMSO

E24-P24	DMSO

A24	Nocodazole

B24	Nocodazole

C24	Tetrandrine

D24	Tetrandrine




In [ ]:
# First lets define a test set
# Batches of full replicate sets that work 
# Batches of full replicate sets that do not work in the qc

In [ ]:
# need a more flexible loader... Best to just create an index of all files with the metadata

In [ ]:
site_specific_path = os.path.join(input_path, "IMTM")
     
# load raw data
pattern = "[A-Z]*_R[1-4].csv"
file_list = glob.glob(site_specific_path + os.sep + '*' + os.sep + pattern, recursive=True)

file_df = pd.DataFrame({'filepath': file_list})

# Extract information using regex
pattern = r'(\d{4}-\d{2}-\d{2})_([A-Z]\d+[A-Z]?)_(R\d+)'

file_df[['batch_date', 'plate_name', 'replicate']] = file_df['filepath'].str.extract(pattern)

print(file_df)

In [ ]:
file_df['batch_date'] = pd.to_datetime(file_df['batch_date'])

file_df['batch_date_yymmdd'] = file_df['batch_date'].dt.strftime('%y%m%d')

In [ ]:
good_plates = [
    ["C1233", "R1", 250114],
    ["C1233", "R2", 240605],
    ["C1233", "R3", 250110],
    ["C1233", "R4", 240605],
    ["C1247", "R1", 240703],
    ["C1247", "R2", 240710],
    ["C1247", "R3", 240711],
    ["C1247", "R4", 240712],
    ["C1256", "R1", 240717],
    ["C1256", "R2", 240724],
    ["C1256", "R3", 240725],
    ["C1256", "R4", 240814],
    ["C1263", "R1", 240724],
    ["C1263", "R2", 240814],
    ["C1263", "R3", 240815],
    ["C1263", "R4", 240816],
]

good_plates_df = pd.DataFrame(good_plates, columns=["plate_name", "replicate", "batch_date_yymmdd"])

In [ ]:
bad_plates = [
    # 251113
    ["C1287", "R3", 251113],
    ["C1288", "R3", 251113],

    # 251118
    ["C1283", "R2", 251118],
    ["C1284", "R2", 251118],
    ["C1285", "R2", 251118],
    ["C1287", "R2", 251118],
    ["C1288", "R2", 251118],

    # 251210
    ["C1287", "R4", 251210],
    ["C1288", "R4", 251210],

    # 251212
    ["C1287", "R1", 251212],
    ["C1288", "R1", 251212],
]

bad_plates_df = pd.DataFrame(bad_plates, columns=["plate_name", "replicate", "batch_date_yymmdd"])

In [ ]:
# Pick one dtype for join keys (string is safest)
for df in (file_df, good_plates_df, bad_plates_df):
    df["plate_name"] = df["plate_name"].astype(str).str.strip()
    df["replicate"] = df["replicate"].astype(str).str.strip()
    df["batch_date_yymmdd"] = df["batch_date_yymmdd"].astype(str).str.strip()

good_plates_files = file_df.merge(
    good_plates_df,
    on=["plate_name", "replicate", "batch_date_yymmdd"],
    how="inner"
)

bad_plates_files = file_df.merge(
    bad_plates_df,
    on=["plate_name", "replicate", "batch_date_yymmdd"],
    how="inner"
)

good_file_list = good_plates_files["filepath"].dropna().tolist()
bad_file_list = bad_plates_files["filepath"].dropna().tolist()

In [ ]:
def read_reduced_file(file_list):

    Data = []

    # Read in the file dataframes 
    for file in file_list:
        
        plate_map_name = os.path.splitext(os.path.basename(file))[0]
        
        try:
            
            Data_Temp = pd.read_csv(file)
            row_count = Data_Temp.shape[0]
            
            print(f"File: {plate_map_name} has {row_count} rows")
            
            Data.append(Data_Temp)
                
        except Exception as e:
            
            print(f"Error reading file {plate_map_name}: {e}")
            
            
    ### concat all files together
    plates_aggregated = pd.concat(Data)
    plates_aggregated = plates_aggregated.reset_index(drop = True)

    print("Aggregated Data has shape ", plates_aggregated.shape)

    cols_to_keep = [
        "Metadata_Batch",
        "Metadata_staining_date",
        "Metadata_Plate",
        "Metadata_Well_randomized",
        "Metadata_Object_Count",
        "Metadata_plate_name",
        "Metadata_replicate_number",
        "Metadata_plate_map_name",
        "Metadata_Well",
        "Metadata_RoughID"
    ]

    plates_aggregated_reduced = plates_aggregated[cols_to_keep].copy()

    # get row and column values 
    plates_aggregated_reduced_copy = plates_aggregated_reduced .copy()

    # Well like "A01" -> row="A", col=1
    plates_aggregated_reduced_copy["row"] = plates_aggregated_reduced_copy["Metadata_Well"].str[0]                 # "A"
    plates_aggregated_reduced_copy["col"] = plates_aggregated_reduced_copy["Metadata_Well"].str[1:].astype(int)    # "01" -> 1

    return plates_aggregated_reduced_copy

In [ ]:
good_plates_aggregated_reduced = read_reduced_file(good_file_list)

# get good min max for the same color scale across all tested plates
color_min = good_plates_aggregated_reduced["Metadata_Object_Count"].min()
color_max = good_plates_aggregated_reduced["Metadata_Object_Count"].max()

In [ ]:
def plot_plate_heatmaps_all_plates(
    df,
    vmin=None,
    vmax=None,
    plate_col="Metadata_plate_name",
    rep_col="Metadata_replicate_number",
    batch_col="Metadata_staining_date",
    row_col="row",
    col_col="col",
    value_col="Metadata_Object_Count",
    replicates=("R1", "R2", "R3", "R4"),
):
    row_order = list("ABCDEFGHIJKLMNOP")  # 16
    col_order = list(range(1, 25))        # 24

    # Ensure replicate ordering (so sorting works even if it's a categorical)
    d = df.copy()
    d[rep_col] = d[rep_col].astype(str)
    d[plate_col] = d[plate_col].astype(str)

    # Sort by plate then replicate (R1..R4)
    rep_rank = {r: i for i, r in enumerate(replicates)}
    d["_rep_rank"] = d[rep_col].map(rep_rank).fillna(999).astype(int)
    d = d.sort_values([plate_col, "_rep_rank"])

    for plate, dplate in d.groupby(plate_col, sort=False):
        fig, axes = plt.subplots(1, len(replicates), figsize=(4 * len(replicates), 3), constrained_layout=True)
        axes = np.atleast_1d(axes).ravel()

        last_im = None

        for ax, rep in zip(axes, replicates):
            drep = dplate[dplate[rep_col] == rep]

            if drep.empty:
                # Keep panel but show it's missing
                ax.set_title(f"{rep} (missing)")
                ax.set_xticks([])
                ax.set_yticks([])
                ax.axis("off")
                continue

            grid = (
                drep.pivot(index=row_col, columns=col_col, values=value_col)
                    .reindex(index=row_order, columns=col_order)
            )

            im = ax.imshow(grid.to_numpy(), vmin=vmin, vmax=vmax, aspect="auto", origin="upper")
            last_im = im

            # Get batch (if multiple, show the first; adjust if you prefer "unique list")
            batch = drep[batch_col].iloc[0] if batch_col in drep.columns else "NA"
            
            ax.text(
                0.01, 1.02,
                f"batch: {batch} | plate: {plate} | rep: {rep}",
                transform=ax.transAxes,
                ha="left", va="bottom",
                fontsize=8
            )

            ax.set_yticks(range(len(row_order)))
            ax.set_yticklabels(row_order)

            ax.set_xticks(range(len(col_order)))
            ax.set_xticklabels([f"{c:02d}" for c in col_order], rotation=90)

        if last_im is not None:
            cbar = fig.colorbar(last_im, ax=axes.tolist(), shrink=0.85)
            cbar.set_label(value_col)

        fig.suptitle(f"Object count heatmaps — {plate}", y=1.05, fontsize=12)
        plt.show()

    # cleanup helper column
    # (if you don't want side effects, keep it in the copy `d` only like here)

In [ ]:
plot_plate_heatmaps_all_plates(good_plates_aggregated_reduced,color_min,color_max)

In [ ]:
bad_plates_aggregated_reduced = read_reduced_file(bad_file_list)

# get good min max for the same color scale across all tested plates
color_min = good_plates_aggregated_reduced["Metadata_Object_Count"].min()
color_max = good_plates_aggregated_reduced["Metadata_Object_Count"].max()

plot_plate_heatmaps_all_plates(bad_plates_aggregated_reduced ,color_min,color_max)

In [ ]:
# There are two different problems. 
# There are the degraded plates. Seems there are missing wells? All in the same pattern... strange
# C1283, C1284, C1285 
# In the others the positive control pattern is off... 
# But also there C1288 R1 is also a different problem with reduced cell number

In [ ]:
# Lets first look into first problem
plate_c1285 = bad_plates_aggregated_reduced[bad_plates_aggregated_reduced["Metadata_plate_name"] == "C1285"]

plt.figure(figsize=(6,4))
plt.hist(plate_c1285["Metadata_Object_Count"], bins=30)
plt.xlabel("your_column_name")
plt.ylabel("Frequency")
plt.title("Histogram of your_column_name")
plt.show()
# First issue is a processing problem.
# There are truncated raw files in the folder. 
# Deleting these truncated files and repeat processing. Re check quality
# Fixed by deleting the truncated files and reprocessing them.

In [ ]:
# Second issue has nothing to do with truncated files
# Could be wrongly pipetted motherplate > Thus positive control position is wrong
# But its from 2 specific plates from different batches that are pipetted just wrong? How likely is that going to be?
# It testable. So we might want to try it and compare correlations
# But what else could this be then?

In [ ]:
# One thing I still find concerning is that in the other fixed plates of the later batch dates there is a strange consistent pattern
# Thus check the last batches and see if they are having this pattern
# Check also the last batches
good_batches = ["250108", "250114", "250423", "250108", "250415"]
bad_batches = ["251022", "251113", "251118", "251210", "251212"]

In [ ]:
good_batches_file_df = file_df[file_df["batch_date_yymmdd"].astype(str).isin(good_batches)]
good_batches_list = good_batches_file_df["filepath"].dropna().tolist()

bad_batches_file_df = file_df[file_df["batch_date_yymmdd"].astype(str).isin(bad_batches)]
bad_batches_list = bad_batches_file_df["filepath"].dropna().tolist()

In [ ]:
good_batches_df = read_reduced_file(good_batches_list)

In [ ]:
# get good min max for the same color scale across all tested plates
color_min = good_batches_df["Metadata_Object_Count"].min()
color_max = good_batches_df["Metadata_Object_Count"].max()

In [ ]:
plot_plate_heatmaps_all_plates(good_batches_df,color_min,color_max)

In [ ]:
bad_batches_df = read_reduced_file(bad_batches_list)

# get good min max for the same color scale across all tested plates
color_min = bad_batches_df["Metadata_Object_Count"].min()
color_max = bad_batches_df["Metadata_Object_Count"].max()

plot_plate_heatmaps_all_plates(bad_batches_df,color_min,color_max)